# Module 19 — Graph Neural Networks (Simplified): Money-Mule Ring Detection

> **Track 2 · Revolut Fintech Specialization**  
> **Difficulty**: Intermediate → Advanced  
> **Runtime**: ~5 minutes  
> **Key Libraries**: NetworkX, Scikit-learn, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 12 (Isolation Forest), Module 13 (Logistic Regression)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Transaction Graph Generation](#3-synthetic-transaction-graph-generation)
4. [Graph Feature Engineering](#4-graph-feature-engineering)
5. [Mule Ring Detection Model](#5-mule-ring-detection-model)
6. [Model Training](#6-model-training)
7. [Model Evaluation](#7-model-evaluation)
8. [Visualization](#8-visualization)
9. [Key Business Takeaway](#9-key-business-takeaway)

---
## §1 · Business Context

### Why detect money-mule rings?

Money laundering through **mule accounts** is a major fintech risk:
- **Circular flows**: Money moves A → B → C → A to obscure origin.
- **Layering**: Multiple hops make tracing difficult.
- **Account takeover**: Innocent accounts used without knowledge.

**Detection approaches**:
1. **Graph features**: Degree centrality, circular flow detection.
2. **GNN approaches**: Message-passing neural networks (simplified here).
3. **Rule-based**: Velocity checks, country risk scores.

**Business impact**:
- Each mule ring launders £50K-500K on average.
- Regulatory fines for missing mule rings: £100K-1M per incident.
- Early detection prevents downstream fraud and compliance issues.

In this notebook we will:
1. Generate a synthetic transaction graph with mule rings.
2. Extract graph features (degree, flow patterns).
3. Train a model to detect mule accounts.
4. Visualize suspicious transaction patterns.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import time
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Graph libraries ──────────────────────────────────────────────
import networkx as nx

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, confusion_matrix, classification_report
)
from sklearn.ensemble import RandomForestClassifier

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Transaction Graph Generation

In [ ]:
# Generate synthetic transaction graph
N_ACCOUNTS = 5000
N_TRANSACTIONS = 20000
N_MULE_RINGS = 20

# Create accounts
accounts = pd.DataFrame({
    'account_id': range(N_ACCOUNTS),
    'account_type': np.random.choice(['personal', 'business', 'mule'], N_ACCOUNTS, p=[0.85, 0.10, 0.05]),
    'age_days': np.random.exponential(365, N_ACCOUNTS).astype(int),
    'country': np.random.choice(['GB', 'US', 'DE', 'FR', 'NG', 'RU'], N_ACCOUNTS, p=[0.4, 0.2, 0.15, 0.1, 0.1, 0.05])
})

# Generate mule ring structures
G = nx.DiGraph()
for i in range(N_ACCOUNTS):
    G.add_node(i)

# Create circular money flows (mule rings)
for ring in range(N_MULE_RINGS):
    ring_size = np.random.randint(3, 8)
    ring_accounts = np.random.choice(N_ACCOUNTS, ring_size, replace=False)
    
    for j in range(ring_size):
        src = ring_accounts[j]
        dst = ring_accounts[(j + 1) % ring_size]
        amount = np.random.uniform(500, 5000)
        G.add_edge(src, dst, amount=amount, is_fraud=1)

# Add normal transactions
for _ in range(N_TRANSACTIONS - N_MULE_RINGS * 5):
    src = np.random.randint(0, N_ACCOUNTS)
    dst = np.random.randint(0, N_ACCOUNTS)
    if src != dst:
        amount = np.random.exponential(200)
        G.add_edge(src, dst, amount=amount, is_fraud=0)

print(f"✓ Generated transaction graph:")
print(f"  Nodes (accounts): {G.number_of_nodes():,}")
print(f"  Edges (transactions): {G.number_of_edges():,}")
print(f"  Mule rings: {N_MULE_RINGS}")

---
## §4 · Graph Feature Engineering

In [ ]:
# Extract graph features for each account
def extract_graph_features(G, node):
    """Extract structural features from the graph."""
    in_edges = list(G.in_edges(node, data=True))
    out_edges = list(G.out_edges(node, data=True))
    
    features = {
        'in_degree': len(in_edges),
        'out_degree': len(out_edges),
        'total_degree': len(in_edges) + len(out_edges),
        'in_amount': sum(e[2].get('amount', 0) for e in in_edges),
        'out_amount': sum(e[2].get('amount', 0) for e in out_edges),
        'avg_in_amount': np.mean([e[2].get('amount', 0) for e in in_edges]) if in_edges else 0,
        'avg_out_amount': np.mean([e[2].get('amount', 0) for e in out_edges]) if out_edges else 0,
    }
    
    # Circular flow detection (simplified)
    features['potential_circular'] = 1 if (features['in_degree'] > 0 and features['out_degree'] > 0) else 0
    
    return features

# Extract features for all accounts
print("Extracting graph features...")
graph_features = []
for node in G.nodes():
    feat = extract_graph_features(G, node)
    feat['account_id'] = node
    graph_features.append(feat)

df_features = pd.DataFrame(graph_features)
df = accounts.merge(df_features, on='account_id')

print(f"✓ Extracted {len(df_features.columns)-1} graph features")
print(f"  Dataset shape: {df.shape}")

---
## §5 · Mule Ring Detection Model

In [ ]:
# Prepare features for model
feature_cols = ['in_degree', 'out_degree', 'total_degree', 'in_amount', 'out_amount',
                'avg_in_amount', 'avg_out_amount', 'potential_circular']

# Label mule accounts (those in circular flows)
df['is_mule'] = (df['account_type'] == 'mule').astype(int)

X = df[feature_cols]
y = df['is_mule']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {len(y_train):,} ({y_train.sum():,} mules)")
print(f"Test set: {len(y_test):,} ({y_test.sum():,} mules)")

---
## §6 · Model Training

In [ ]:
# Train a simple classifier
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train_scaled, y_train)

# Cross-validation
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='roc_auc')
print(f"✓ Model trained")
print(f"  5-Fold CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

---
## §7 · Model Evaluation

In [ ]:
# Predictions
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# Metrics
roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print("=" * 70)
print("MULE RING DETECTION MODEL EVALUATION")
print("=" * 70)
print(f"\nROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Mule']))

---
## §8 · Visualization

In [ ]:
# Feature importance
importance = model.feature_importances_
feat_imp = pd.DataFrame({'Feature': feature_cols, 'Importance': importance}).sort_values('Importance', ascending=False)

fig = px.bar(feat_imp, x='Importance', y='Feature', orientation='h',
             title='Feature Importance for Mule Detection',
             color='Importance', color_continuous_scale='Viridis')
fig.update_layout(height=400, yaxis=dict(autorange='reversed'))
fig.show()

print("💡 Key insights:")
print("   - Circular flow indicators are top predictors")
print("   - Degree centrality helps identify suspicious accounts")
print("   - Transaction amounts reveal anomalous patterns")

---
## §9 · Key Business Takeaway

> ### 🎯 Action Items for Fintech Fraud Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Money laundering detection, mule account identification |
> | **Features** | Graph structural features (degree, flow patterns) |
> | **Model** | Random Forest or GNN for graph-based classification |
> | **Evaluation** | PR-AUC for imbalanced detection |
> | **Deployment** | Real-time graph analysis on transaction networks |
>
> ### 💡 Revolut-Specific Guidelines
>
> 1. **Build transaction graphs in real-time**:
>    ```
>    Transaction → Update graph → Extract features → Score account
>    ```
>
> 2. **Monitor circular flows**:
>    | Pattern | Risk Level | Action |
>    |---------|------------|--------|
>    | A → B → A | High | Freeze accounts, investigate |
>    | A → B → C → A | Critical | Report to authorities |
>    | Long chains | Medium | Enhanced monitoring |
>
> 3. **Combine with other signals**:
>    - Velocity checks (rapid transactions)
>    - Geographic anomalies (high-risk countries)
>    - Amount patterns (structuring, round amounts)
>
> ### 🔑 Key Takeaways
>
> 1. **Graph features reveal hidden patterns** invisible to traditional models.
> 2. **Circular flows are strong indicators** of money laundering.
> 3. **Degree centrality** helps identify hub accounts in mule networks.
> 4. **Real-time graph updates** enable instant detection.
> 5. **Combine graph ML with rule-based systems** for comprehensive coverage.